In [1]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("data/audi.csv")
dep_df = pd.read_csv("data/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# AUDI MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "audi-a3": "A3",
    "audi-a4": "A4",
    "audi-a5": "A5",
    "audi-a6": "A6",
    "audi-a7": "A7",
    "audi-a8": "A8",
    "audi-e-tron": "E-TRON",
    "audi-q2": "Q2",
    "audi-q3": "Q3",
    "audi-q4-e-tron": "Q4 E-TRON",
    "audi-q5": "Q5",
    "audi-q7": "Q7",
    "audi-q8": "Q8",
    "audi-q8-e-tron": "Q8 E-TRON",
    "audi-r8": "R8",
    "audi-rs3": "RS3",
    "audi-rs4": "RS4",
    "audi-rs5": "RS5",
    "audi-rs6": "RS6",
    "audi-rs7": "RS7",
    "audi-rsq3": "RS Q3",
    "audi-rsq8": "RS Q8",
    "audi-s3": "S3",
    "audi-s4": "S4",
    "audi-s5": "S5",
    "audi-s6": "S6",
    "audi-s7": "S7",
    "audi-s8": "S8",
    "audi-sq5": "SQ5",
    "audi-sq7": "SQ7",
    "audi-sq8": "SQ8",
    "audi-tt": "TT",
}

df["make"] = "AUDI"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "data/audi_with_depreciation_annual_rate.csv",
    index=False
)

print("\nSaved: data/audi_with_depreciation_annual_rate.csv")

Total Rows: 935
Matched Rates: 626
Missing Rates: 309

Unmatched Models:
['audi-a1', 'audi-a3-sedan', 'audi-a5-cabriolet', 'audi-a5-coupe', 'audi-a5-sportback', 'audi-a6-sportback-e-tron', 'audi-e-tron-gt-quattro', 'audi-e-tron-sportback', 'audi-q3-sportback', 'audi-q6-e-tron', 'audi-q8-e-tron-sportback', 'audi-r8-spyder', 'audi-rs-3', 'audi-rs-3-sedan', 'audi-rs-4', 'audi-rs-5', 'audi-rs-5-sportback', 'audi-rs-6', 'audi-rs-7', 'audi-rs-e-tron-gt', 'audi-rs-q3', 'audi-rs-q8', 'audi-s-q5', 'audi-s-q7', 'audi-s-q8', 'audi-s3-sportback', 'audi-sq8-e-tron-sportback']

Saved: data/audi_with_depreciation_annual_rate.csv
